# 01 — Camada Bronze: Ingestão

Lê os arquivos brutos da CVM depositados no Volume do Unity Catalog e os persiste como
tabelas Delta, **sem nenhuma transformação de conteúdo**.

**Princípios desta camada**

| Decisão | Motivo |
|---|---|
| Todos os campos lidos como texto | Tipagem é transformação explícita e pertence à silver. Inferir tipos aqui apagaria a evidência do que efetivamente chegou |
| Leitura arquivo a arquivo, unida por nome de coluna | Protege contra divergência de layout entre meses, risco real dada a migração para a Resolução CVM 175 |
| Metadados de controle em cada linha | Rastreabilidade: de qual arquivo veio e quando foi ingerido |
| Nada é filtrado ou deduplicado | A bronze é o cofre de evidências |

**Fonte:** Portal de Dados Abertos da CVM — licença ODbL (Open Data Commons).

In [ ]:
from pyspark.sql import functions as F
from functools import reduce
import zipfile, os

VOL = "/Volumes/workspace/bronze/raw_cvm"
DST = VOL + "/csv"
ENCODING = "ISO-8859-1"   # arquivos da CVM são latin-1, não UTF-8
SEP = ";"

os.makedirs(DST, exist_ok=True)

## 1.1 Descompactação

Os informes são distribuídos em ZIPs mensais. A descompactação acontece dentro do próprio
Volume, preservando os arquivos originais ao lado dos extraídos.

In [ ]:
arquivos = []
for nome in sorted(os.listdir(VOL)):
    if nome.lower().endswith(".zip"):
        z = zipfile.ZipFile(os.path.join(VOL, nome))
        for m in z.namelist():
            if m.lower().endswith(".csv"):
                z.extract(m, DST)
                arquivos.append(os.path.join(DST, m))
        z.close()

print(len(arquivos), "arquivos CSV disponiveis")
for a in sorted(arquivos):
    print("  ", os.path.basename(a))

## 1.2 Leitura

Cada arquivo é lido isoladamente e depois unido pelos nomes das colunas. Ler a pasta inteira
de uma vez seria mais curto, mas desalinharia os dados caso o layout mudasse entre meses.
`allowMissingColumns=True` preenche com nulo em vez de quebrar.

In [ ]:
def ler(caminho, origem_literal=None):
    df = (spark.read
          .option("header", True)
          .option("sep", SEP)
          .option("encoding", ENCODING)
          .option("inferSchema", False)
          .csv(caminho))
    origem = F.lit(origem_literal) if origem_literal else F.col("_metadata.file_path")
    return (df.withColumn("_arquivo_origem", origem)
              .withColumn("_data_ingestao", F.current_timestamp()))

def unir(caminhos):
    return reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True),
                  [ler(c) for c in caminhos])

## 1.3 Informe Diário

Grão: classe/subclasse por dia útil. Onze meses, de set/2025 a jul/2026.

In [ ]:
infs = [c for c in arquivos if "inf_diario" in os.path.basename(c).lower()]
df_inf = unir(infs)

(df_inf.write.format("delta").mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("workspace.bronze.inf_diario_fi"))

print("inf_diario_fi:", df_inf.count(), "linhas")
print("colunas:", spark.table("workspace.bronze.inf_diario_fi").columns)

## 1.4 Cadastro: fundo, classe e subclasse

O ZIP do registro traz três arquivos, refletindo a hierarquia da Resolução CVM 175:
fundo (casca) → classe → subclasse.

In [ ]:
z = zipfile.ZipFile(VOL + "/registro_fundo_classe.zip")
z.extractall(DST)
z.close()

for arq, tab in [("registro_fundo.csv", "cad_fundo"),
                 ("registro_classe.csv", "cad_classe"),
                 ("registro_subclasse.csv", "cad_subclasse")]:
    d = ler(DST + "/" + arq, origem_literal=arq)
    (d.write.format("delta").mode("overwrite")
       .option("overwriteSchema", True)
       .saveAsTable("workspace.bronze." + tab))
    print(tab, "->", d.count(), "linhas")

## 1.5 Conferência da ingestão

Contagem por arquivo de origem: confirma que nenhum mês ficou de fora e detecta arquivo
truncado. A variação entre meses é explicada por dias úteis e pelo crescimento no número
de classes ativas.

In [ ]:
display(
    spark.table("workspace.bronze.inf_diario_fi")
      .groupBy(F.element_at(F.split("_arquivo_origem", "/"), -1).alias("arquivo"))
      .count()
      .orderBy("arquivo")
)